# 01 — Phase 1: PDF Download + YOLO Crop Extraction
Reads the Excel, downloads PDFs, runs DocLayout-YOLO, saves crops.

**Before running:** check config.yaml → subset.mode to control which patents to process.

In [ ]:
# ── CELL 1: Setup ────────────────────────────────────────────────
import sys; sys.path.insert(0, "..")
import yaml
from src.patents import load_patents, get_subset
from src.downloader import download_pdfs
from src.extractor import extract_crops

with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

print("Config loaded.")
print(f"  Subset mode : {cfg['subset']['mode']}")
print(f"  YOLO device : {cfg['extractor']['device']}")
print(f"  PDF DPI     : {cfg['extractor']['pdf_dpi']}")

---
**Optional:** If you ran  and saved a list, load it here instead of using the config subset mode.

In [ ]:
# ── CELL 2: Load patents ──────────────────────────────────────────
# Option A: use config.yaml subset settings (default)
df      = load_patents(cfg)
subset  = get_subset(df, cfg)

# Option B: load from saved filter list (uncomment to use)
# from pathlib import Path
# ids = Path("../selected_patents.txt").read_text().strip().splitlines()
# subset = df[df["Record Number"].isin(ids)].copy()
# print(f"Loaded {len(subset)} patents from saved filter")

print(f"
Ready to process {len(subset)} patents.")
subset[["Record Number", "Title"]].head(5)

In [ ]:
# ── CELL 3: Download PDFs ────────────────────────────────────────
# Skips already-downloaded files automatically
pdf_paths = download_pdfs(subset, cfg)

In [ ]:
# ── CELL 4: Extract crops with DocLayout-YOLO ────────────────────
crop_results = extract_crops(pdf_paths, cfg)

# Summary
total_crops = sum(len(v) for v in crop_results.values())
empty = [k for k,v in crop_results.items() if len(v)==0]
print(f"
Total crops: {total_crops}")
print(f"Patents with 0 crops: {len(empty)}")
if empty:
    print("  ", empty[:10])

In [ ]:
# ── CELL 5: Save crop index for Phase 2 ─────────────────────────
import json
from pathlib import Path

crop_index = {k: [str(p) for p in v] for k, v in crop_results.items()}
out = Path(cfg["paths"]["logs"]) / "crop_index.json"
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w") as f:
    json.dump(crop_index, f, indent=2)
print(f"Crop index saved to: {out}")